In [1]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv(), override=True)

True

In [2]:
# pip install docx2txt -q
!pip install wikipedia -q

In [3]:
def load_document(file):
    import os
    name , extension = os.path.splitext(file)
    
    if extension == ".pdf":
        from langchain.document_loaders import PyPDFLoader
        print(f"Loading {file}")
        loader = PyPDFLoader(file)
        
    elif extension == ".docx":
        from langchain.document_loaders import Docx2txtLoader
        print(f"Loading {file}")
        loader = Docx2txtLoader(file)
        
    else:
        return None
        
    
        
        
        
        
    data = loader.load()
    return data

## Runing Code

In [4]:
data = load_document("files/us_constitution.pdf")
print(data[1].metadata)

Loading files/us_constitution.pdf
{'source': 'files/us_constitution.pdf', 'page': 1}


In [5]:
print(f"There are {len(data)} Pages")
print(f"Number of charaters in the page {len(data[0].page_content)}")

There are 41 Pages
Number of charaters in the page 617


### Docs

In [6]:
data = load_document("files/the_great_gatsby.docx")
print(data[0].metadata)

Loading files/the_great_gatsby.docx
{'source': 'files/the_great_gatsby.docx'}


In [7]:
print(f"There are {len(data)} Pages")
print(f"Number of charaters in the page {len(data[0].page_content)}")

There are 1 Pages
Number of charaters in the page 296672


### Private or Public service

In [41]:
 # wikipedia 

def load_from_wikipedia(query, lang='en'):
    from langchain.document_loaders import WikipediaLoader
    
    loader = WikipediaLoader(query=query, lang=lang, load_max_docs=10)
    
    data = loader.load()
    
    return data

In [42]:
data = load_from_wikipedia("استراتيجيات للوقاية من مرض القلب",'ar')

In [43]:
print(data[0].page_content)

القصور القلبي
(بالإنجليزية: cardiac insufficiency)‏  أو فشل القلب (بالإنجليزية: Heart failure)‏ هو حالة مرضية تحدث نتيجة أي خلل وظيفي أو عضوي يؤثر على وظيفة القلب كمستقبل للدم ومضخة كافية لانتشار الدم إلى أنحاء الجسم لإمداده بالمواد الغذائية والأكسجين. تشمل العلامات والأعراض عادةً ضيقًا في التنفس، التعب المفرط، وتورُّم الساق. عادةً ما يكون ضيق التنفس أسوأ مع ممارسة الرياضة، وعند الاستلقاء، حتى أنه قد يوقِظ المريض ليلا. بينما لا يحدث عادة ألم الصدر، بما في ذلك الذبحة الصدرية، بسبب فشل القلب.
من الأسباب الشائعة لقصور القلب: أمراض الشرايين التاجية بما في ذلك احتشاء عضلة القلب والأشكال الأخرى للداء القلبي الإقفاري، وفرط ضغط الدم، وداء قلبي صمامي، واعتلال عضلة القلب، وأيضًا زيادة تعاطي الكحول. يسبب ذلك قصور القلب عن طريق تغيير تكوين أو وظيفة القلب. لا يُعد فشل القلب هو نفسه احتشاء عضلة القلب (موت جزء من عضلة القلب) أو السكتة القلبية (التي توقِف تدفُّق الدم تمامًا). وتشمل الأمراض الأخرى التي قد يكون لها أعراض مشابهة لفشل القلب: السمنة، الفشل الكلوي، مشاكل الكبد، فقر الدم وأمراض الغدة الدرقية

## Chunking 
it is the process of breaking down large piecese of text into smaller segments

In [44]:
def chunk_data(data, chunk_size=256):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

In [45]:
# data = load_document("files/us_constitution.pdf")
chunks = chunk_data(data)

In [46]:
print(len(chunks))

215


In [47]:
print(chunks[10].page_content)

== أنواع ==


In [48]:
import tiktoken
def printing_cost(texts):
    enc = tiktoken.encoding_for_model('text-embedding-ada-002')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f"Total Tokens is {total_tokens}")
    print(f"Embedding cost in USD {total_tokens / 1000 * 0.0004:0.6f}")

In [49]:
printing_cost(chunks)

Total Tokens is 27728
Embedding cost in USD 0.011091


## Embedding and upload to a vector DB

In [50]:
def insert_of_fetch_embeddings(index_name):
    import pinecone
    from langchain.vectorstores import Pinecone
    from langchain.embeddings.openai import OpenAIEmbeddings
    
    embeddings = OpenAIEmbeddings()
    
    pinecone.init(
    api_key = os.environ.get('PINECODE_API_KEY'),
    environment=os.environ.get("PINECONE_ENV"))
    
    if index_name in pinecone.list_indexes():
        print(f"Index {index_name} already exists")
        vector_store = Pinecone.from_existing_index(index_name, embeddings)
    else:
        print(f"Creating index name")
        pinecone.create_index(index_name, dimension=1536,
                             metric='cosine')
        vector_store = Pinecone.from_documents(chunks, embeddings, index_name=index_name)
        print("ok")
    
    return vector_store

In [51]:
def delete_index(index_name='all'):
    import pinecone
    pinecone.init(
    api_key = os.environ.get('PINECODE_API_KEY'),
    environment=os.environ.get("PINECONE_ENV"))
    if index_name == "all":
        indexes = pinecone.list_indexes()
        print("Deleting All indexes ..")
        for index in indexes:
            pinecone.delete_index(index)
    else:
        
        print(f"Deleting Index {index_name}")
        pinecone.delete_index(index_name) 

In [52]:
delete_index("all")

Deleting All indexes ..


In [53]:
index_name = 'ask-document'
vector_store = insert_of_fetch_embeddings(index_name)

Creating index name
ok


## Asking and Getting Answers

In [54]:
def ask_get_answer(vector_store, question):
    from langchain.chains import RetrievalQA
    from langchain.chat_models import ChatOpenAI
    
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=1)
    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={"k":3})
    chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)
    
    answer = chain.run(question)
    
    return answer

def ask_with_memory(vector_store, question, chat_histoy):
    from langchain.chains import ConversationalRetrievalChain
    from langchain.chat_models import ChatOpenAI
    
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=1)
    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={"k":3})
    
    crc = ConversationalRetrievalChain.from_llm(llm, retriever)
    
    result = crc({"question":question, "chat_history":chat_histoy})
    
    chat_history.append((question,result['answer'] ))
    
    return result, chat_history
    

In [55]:
question = "what is the whole document about"
answer = ask_get_answer(vector_store, question)
print(answer)

The whole document appears to be about reducing the burden of disease worldwide. It discusses the importance of efforts towards a healthier diet, increased physical activity, and the control of non-communicable diseases. It also emphasizes the development of self-awareness and insight for healthcare providers, allowing them to build better relationships with patients and their families. Additionally, the document mentions the importance of understanding the social conditions that can impact individual health. It specifically notes that three-quarters of these deaths occur in the part of Africa located south of the Sahara Desert, which hinders economic growth and depletes human capital.


In [56]:
import time
i = 1
print("Write Quit or Exit to quit")
while True:
    question = input(f"Question #{i}")
    i +=1
    if question.lower() in ['quit', 'exit']:
        print("Quitting ... ")
        time.sleep(2)
        break
    answer = ask_get_answer(vector_store, question)
    print(f"\nAnswer : {answer}")
    print(f"\n {'-' * 50} \n")

Write Quit or Exit to quit


Question #1 كيف يمكن التخفيف من العبء الناجم عن أمراض القلب والأوعية الدموية



Answer : يمكن التخفيف من العبء الناجم عن أمراض القلب والأوعية الدموية عبر اتباع بعض الإجراءات الوقائية والحماية للحفاظ على صحة القلب والأوعية الدموية. قد تتضمن هذه الإجراءات:

1. الاهتمام بالنظام الغذائي: تناول طعام صحي ومتوازن يحتوي على الفواكه والخضروات والألياف والحبوب الكاملة، وتجنب تناول الدهون المشبعة والأطعمة العالية بالصوديوم.

2. ممارسة الرياضة: قم بممارسة التمارين الرياضية بانتظام وبشكل منتظم، وفي حدود قدراتك. يفضل ممارسة الرياضات المعتدلة مثل المشي السريع، وركوب الدراجة، والسباحة.

3. التحكم في الوزن: حافظ على وزن صحي، وضمن النطاق الطبيعي للجسم. السمنة قد تزيد من خطر الإصابة بأمراض القلب والأوعية الدموية.

4. التقليل من التوتر: حاول تقليل مستويات التوتر والضغط النفسي، واستخدام تقنيات الاسترخاء مثل التأمل والتنفس العميق.

5. الامتناع عن التدخين: قم بالامتناع عن التدخين وتجنب التعرض للدخان السلبي. التدخين يزيد من خطر الإصابة بأمراض القلب والأوعية الدموية.

6. اتباع العلاجات الموصوفة: اتبع توجيهات الأطباء واحرص على استخدام الأدوية الموصوفة وإجراءات العلاج الموصوفة للحفاظ على ص

Question #2 quit


Quitting ... 


## Asking With memory

In [35]:
chat_history = []
question = "how many amendments are in the U.S consitution"
result , chat_history = ask_with_memory(vector_store, question, chat_history)
print(result['answer'])
print(chat_history)

There are currently 27 amendments in the U.S. Constitution.
[('how many amendments are in the U.S consitution', 'There are currently 27 amendments in the U.S. Constitution.')]


In [38]:
question = "multiply that number by 2"
result , chat_history = ask_with_memory(vector_store, question, chat_history)
print(result['answer'])
print(chat_history)

The number of amendments in the U.S. Constitution is 27. Multiplying 27 by 2 gives us a result of 54.
[('how many amendments are in the U.S consitution', 'There are currently 27 amendments in the U.S. Constitution.'), ('multiply that number by 2', "Sorry, I don't have enough information to answer your question."), ('multiply that number by 2', 'The number of amendments in the U.S. Constitution is 27. Multiplying 27 by 2 gives us a result of 54.')]
